# rpx：显式映射与可检查的证明

模型与表达式在 Rust 中携带空间身份。两个方向都完成后才编译；Kernel 保持 MCM 1、RPA 12、RPX 11。

## Offline help
Use `python -m rpx help quickstart` for a self-contained first proof,
`python -m rpx help agent` for task navigation, and `help(Scope.map)`
after importing Scope for an exact API contract. Help ships in the wheel.
Run `python -m rpx._help.examples.lifts` for all five verified lift examples.


In [ ]:
from pathlib import Path
import sys, tempfile
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "crates/compiler-core").exists())
sys.path.insert(0, str(root / "python"))
from rpx import Model, ModelBuilder, Proof, Q, quicksum, Bundle, Chain, CoordinateSpaceError, ScopeError


## 从文件开始

稀疏映射只列改动的输出变量。默认匹配由 Rust 按名称完成。

In [ ]:
base = root / "docs/rpx/examples/elimination"
source, target = Model.read(base / "source.lp"), Model.read(base / "target.lp")
source


In [ ]:
proof = Proof(source, target)
f = proof.forward.map({"y": source.vars["x1"]}, default="same_name")
f.plan()


In [ ]:
f.prove("objective.non_increasing", by=-f.row("balance"))
f.finish()
proof.reverse.map({"x0": 1-target.vars["y"], "x1": target.vars["y"]}).finish()
proof.preflight()


## 自包含交付

包内保留模型、两个证明格式和目标原文件。每次 verify 都重新使用本次预算调用 Kernel，并独立审计 artifact。

In [ ]:
workspace = tempfile.TemporaryDirectory(prefix="rpx-tutorial-")
bundle = proof.compile(Path(workspace.name) / "elimination", target_artifact=base / "target.lp")
report = bundle.verify()
report.raise_for_status()
report


## 原生精确数与模型构造

接受整数、精确字符串、Fraction、有限 Decimal。float 和 bool 被拒绝。build 后仍可使用原句柄；模型编辑生成新空间。

In [ ]:
builder = ModelBuilder("box")
x = builder.add_var("x", integer=True, lower=0, upper=1)
y = builder.add_var("y", lower=0, upper=1)
builder.add_row("capacity", x+y, upper=1)
builder.minimize(x/3 + y/7)
model = builder.build()
assert quicksum(model.vars) == x+y
model


In [ ]:
edited = model.with_bounds("y", upper=2)
try:
    x + edited.vars["x"]
except CoordinateSpaceError as error:
    print(error)


## 分支与构造扩展

连续 split 的右分支为 expr ≥ at；整数 split 使用相隔 1 的阈值并要求整性证明。扩展父级固定后，在返回的 child scope 完成映射。

In [ ]:
p = Proof(model, model)
left, right = p.forward.split(x, kind="integer", because=p.forward.integer("x"))
left.map(default="same_name").finish()
right.map(default="same_name").finish()
lift = p.reverse.lift_interval(lower=0, upper=y, because=p.reverse.lower("y"))
lift.scope.map(default="same_name").finish()
p.verify(Path(workspace.name) / "branches").raise_for_status()


## Cases：两个显式推导 scope

分别提交共同结论，再将结果返回父 scope。Fact 算术构造 LinCombination，与 Expr 分离。

In [ ]:
p = Proof(model, model)
s = p.forward
cases = s.derive.cases(x, kind="Le", expect=x-1, integer=s.integer("x"))
cases.left.conclude(cases.left.derive.le(cases.left.assumption-1))
cases.right.conclude(cases.right.upper("x"))
fact = cases.finish()
s.map(default="same_name").finish()
p.reverse.map(default="same_name").finish()
p.verify(Path(workspace.name) / "cases").raise_for_status()


## 诊断与封闭状态

映射更改使旧 Goal 失效。finish 成功后不能追加步骤；需要重新构建证明。视图默认有界显示，整体物化必须显式调用 to_dict / to_list。

In [ ]:
try:
    s.derive.le(s.lower("x"))
except ScopeError as error:
    print(error)
from rpx.lowlevel import CertificateView
proof.certificate()


In [ ]:
workspace.cleanup()